In [ ]:
from google.colab import files

In [ ]:
uploaded=files.upload()

Saving skin_dataset_10000_plus_original_preserved.csv to skin_dataset_10000_plus_original_preserved (1).csv


In [ ]:
import pandas as pd
df=pd.read_csv("skin_dataset_10000_plus_original_preserved.csv")


In [ ]:
df.head()

,text,skin_type,oiliness,dryness,sensitivity
0,"Most days, I have oily areas around my forehea...",Combination,High,Medium,Medium
1,I need a lightweight product for my oily T-zon...,Combination,High,Medium,Medium
2,"Recently, My skin usually lacks moisture and c...",Dry,Low,Medium,Low
3,"In general, I feel like my skin gets oily even...",Oily,High,Medium,Low
4,"In general, My cheeks sometimes feel tight eve...",Combination,Medium,Medium,Low


In [ ]:
df.value_counts("text")

,count
text,
"Recently, My face becomes dry easily when I spend time in air conditioning.",15
"In general, I do not notice much change in my skin during the day.",11
I find that I rarely notice shine on my face and sometimes have dry patches.,11
"In general, My skin produces a lot of oil during the day, especially around my nose.",11
"I find that After washing my face, it feels stretched and dry for a while.",11
...,...
I've found that my nose and forehead are shiny for much of the day. I can generally try a new product without a reaction. I have to reapply moisturizer because my skin feels parched.,1
"I've found that my nose and forehead are shiny for much of the day, although dryness is noticeable most days, not just in winter. On top of that, even products made for sensitive skin can irritate my face. I notice it more after washing my face.",1
I've found that my makeup slips because my face gets so oily. I rarely notice redness after washing my face. I need a rich moisturizer to keep my face comfortable.,1


In [ ]:
counts = df["text"].value_counts()
df = df[df["text"].map(counts) <= 3]

In [ ]:
df["text"].value_counts()

,count
text,
"In general, I rarely have problems with excessive shine or dry patches. This happens quite often.",3
"For me, My nose and forehead get oily very quickly.",3
"Recently, My skin generally feels smooth and balanced.",3
"Most days, My face generally feels comfortable after washing.",3
"I find that After washing my face, it feels stretched and dry for a while. Especially during the day.",3
...,...
"If I'm being honest, my skin feels comfortable rather than oily. When it comes to dryness, my skin generally feels smooth and comfortable. As for reactions, I rarely notice redness after washing my face. It doesn't seem to happen only after using one particular product.",1
"I've noticed that oil builds up quickly even without skincare products, while my cheeks can feel tight by the end of the day. I also find that I get itchy patches after using lots of different creams. I notice it more after washing my face.",1
"I've found that my nose gets a little shiny around lunchtime. Lately, my skin often feels tight, rough, and uncomfortable. Also, my face reacts to products that other people tolerate. It tends to happen even when I keep my routine simple.",1


In [ ]:
df.duplicated().sum()
df.drop_duplicates()

,text,skin_type,oiliness,dryness,sensitivity
1,I need a lightweight product for my oily T-zon...,Combination,High,Medium,Medium
3,"In general, I feel like my skin gets oily even...",Oily,High,Medium,Low
6,"Recently, My skin generally feels smooth and b...",Normal,Medium,Low,Medium
13,"From my experience, Most skincare products do ...",Normal,Medium,Low,Low
14,"For me, My face becomes dry easily when I spen...",Dry,Low,High,Medium
...,...,...,...,...,...
9995,"Quite often, I rarely have that slippery feeli...",Dry,Low,Medium,Low
9996,"For some reason, I notice some oil around my n...",Combination,Medium,High,High
9997,"As far as I can tell, my skin gets greasy even...",Combination,High,High,Low
9998,"More often than not, I almost never reach for ...",Dry,Low,Medium,Medium


In [ ]:
df.shape

(7194, 5)

In [ ]:
!pip install pandas spacy pyspellchecker scikit-learn
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 16.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:



import re
import html
import spacy
import pandas as pd

from spellchecker import SpellChecker
from sklearn.base import BaseEstimator, TransformerMixin


class SkinTextPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(self, correct_spelling=True):
        self.correct_spelling_enabled = correct_spelling

        self.nlp = spacy.load("en_core_web_sm")
        self.spell = SpellChecker()

        self.domain_words = {
            "oily", "oiliness", "oilier", "dry", "dryness",
            "flaky", "flakiness", "sensitive", "sensitivity",
            "sebum", "comedones", "acne", "skincare",
            "moisturizer", "moisturiser", "forehead", "cheek",
            "cheeks", "pores", "irritation", "irritated",
            "hydration", "dehydration", "combination",
            "dermatitis", "noncomedogenic", "greasy", "shiny",
            "tightness", "peeling", "peel", "itchy", "itching",
            "stinging", "sting", "burning", "burn", "redness",
            "red", "patches", "patch", "t-zone", "comedogenic",
            "exfoliation", "exfoliate", "cleanser", "cleansing",
            "fragrance", "fragranced", "moisturizing",
            "moisturising", "dewy", "parched", "slick",
            "scaly", "scaling"
        }

        self.keep_words = {
            "no", "not", "never", "nor", "neither", "without",
            "cannot", "hardly",
            "very", "extremely", "really", "quite", "too",
            "slightly", "somewhat", "mildly", "moderately",
            "barely", "little", "much", "more", "less",
            "most", "least", "enough", "especially",
            "always", "often", "usually", "sometimes",
            "occasionally", "rarely", "seldom", "frequently",
            "constantly", "daily", "weekly", "regularly",
            "but", "however", "although", "though", "whereas",
            "after", "before", "during", "until", "when",
            "while", "if", "because", "than", "then", "that",
            "again", "still", "only", "even"
        }

        self.contractions = {
            "doesn't": "does not",
            "don't": "do not",
            "didn't": "did not",
            "isn't": "is not",
            "aren't": "are not",
            "wasn't": "was not",
            "weren't": "were not",
            "hasn't": "has not",
            "haven't": "have not",
            "hadn't": "had not",
            "can't": "cannot",
            "couldn't": "could not",
            "wouldn't": "would not",
            "shouldn't": "should not",
            "won't": "will not",
            "mustn't": "must not",
            "i'm": "i am",
            "i've": "i have",
            "i'll": "i will",
            "i'd": "i would",
            "it's": "it is",
            "that's": "that is",
            "there's": "there is",
            "they're": "they are",
            "we're": "we are",
            "you're": "you are",
            "you've": "you have",
            "they've": "they have",
            "we've": "we have",
            "what's": "what is",
            "who's": "who is"
        }

        self.protected_words = self.domain_words | self.keep_words

    def basic_cleaning(self, text):
        text = html.unescape(str(text)).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(
            r"\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b", " ", text
        )
        text = re.sub(r"[^a-z'\s-]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def expand_contractions(self, text):
        for contraction, expanded in self.contractions.items():
            text = re.sub(
                rf"\b{re.escape(contraction)}\b",
                expanded,
                text
            )
        return text

    def correct_spelling(self, text):
        if not self.correct_spelling_enabled:
            return text

        corrected_words = []

        for word in text.split():
            if word in self.protected_words or len(word) <= 2:
                corrected_words.append(word)
                continue

            if word not in self.spell:
                suggestion = self.spell.correction(word)

                if suggestion and suggestion.isalpha():
                    corrected_words.append(suggestion.lower())
                else:
                    corrected_words.append(word)
            else:
                corrected_words.append(word)

        return " ".join(corrected_words)

    def tokenize_and_filter(self, text):
        doc = self.nlp(text)
        result = []

        for token in doc:
            if token.is_space or token.is_punct:
                continue

            word = token.text.lower()

            # Preserve words affecting intensity, frequency, or meaning.
            if word in self.protected_words:
                result.append(word)
                continue

            if token.is_stop or not token.is_alpha:
                continue

            # spaCy lemmatizes using its linguistic analysis, including POS.
            lemma = token.lemma_.lower().strip()

            if lemma:
                result.append(lemma)

        return " ".join(result)

    def process_with_steps(self, text):
        steps = {}

        steps["original"] = str(text)

        steps["basic_cleaning"] = self.basic_cleaning(
            steps["original"]
        )

        steps["contraction_expansion"] = self.expand_contractions(
            steps["basic_cleaning"]
        )

        steps["spelling_correction"] = self.correct_spelling(
            steps["contraction_expansion"]
        )

        # Show each token's POS tag and lemma.
        doc = self.nlp(steps["spelling_correction"])

        steps["pos_tags_and_lemmas"] = [
            {
                "word": token.text,
                "pos": token.pos_,
                "lemma": token.lemma_
            }
            for token in doc
            if not token.is_space and not token.is_punct
        ]

        steps["tokenization_stopwords_lemmatization"] = (
            self.tokenize_and_filter(
                steps["spelling_correction"]
            )
        )

        steps["final"] = steps[
            "tokenization_stopwords_lemmatization"
        ]

        return steps

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.Series):
            texts = X.fillna("").astype(str).tolist()
        else:
            texts = pd.Series(X).fillna("").astype(str).tolist()

        return [
            self.process_with_steps(text)["final"]
            for text in texts
        ]



In [ ]:
preprocessor = SkinTextPreprocessor(correct_spelling=True)

sample = (
    "My skin doesn't get very oily, but my cheeks "
    "are extremely dry and flaky after washing."
)

steps = preprocessor.process_with_steps(sample)

for step_name, output in steps.items():
    print(f"\n--- {step_name.upper()} ---")
    print(output)


--- ORIGINAL ---
My skin doesn't get very oily, but my cheeks are extremely dry and flaky after washing.

--- BASIC_CLEANING ---
my skin doesn't get very oily but my cheeks are extremely dry and flaky after washing

--- CONTRACTION_EXPANSION ---
my skin does not get very oily but my cheeks are extremely dry and flaky after washing

--- SPELLING_CORRECTION ---
my skin does not get very oily but my cheeks are extremely dry and flaky after washing

--- POS_TAGS_AND_LEMMAS ---
[{'word': 'my', 'pos': 'PRON', 'lemma': 'my'}, {'word': 'skin', 'pos': 'NOUN', 'lemma': 'skin'}, {'word': 'does', 'pos': 'AUX', 'lemma': 'do'}, {'word': 'not', 'pos': 'PART', 'lemma': 'not'}, {'word': 'get', 'pos': 'VERB', 'lemma': 'get'}, {'word': 'very', 'pos': 'ADV', 'lemma': 'very'}, {'word': 'oily', 'pos': 'ADJ', 'lemma': 'oily'}, {'word': 'but', 'pos': 'CCONJ', 'lemma': 'but'}, {'word': 'my', 'pos': 'PRON', 'lemma': 'my'}, {'word': 'cheeks', 'pos': 'NOUN', 'lemma': 'cheek'}, {'word': 'are', 'pos': 'AUX', 'lemm

In [ ]:
from sklearn.model_selection import train_test_split

# Input text and target labels
X = df["text"].fillna("").astype(str)

targets = ["oiliness", "dryness", "sensitivity"]
y = df[targets]

# First: 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Second: split temporary data into 10% validation and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

Training samples: 5755
Validation samples: 719
Testing samples: 720


In [ ]:
preprocessor = SkinTextPreprocessor()

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# ==========================================
# PIPELINE 1: TF-IDF + LOGISTIC REGRESSION
# ==========================================

lr_pipeline = Pipeline([
    (
        "preprocessor",
        SkinTextPreprocessor(correct_spelling=True)
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=8000,
            sublinear_tf=True,
            min_df=2,
            max_df=0.95
        )
    ),
    (
        "classifier",
        MultiOutputClassifier(
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    )
])


# ==========================================
# PIPELINE 2: TF-IDF + LINEAR SVM
# ==========================================

svm_pipeline = Pipeline([
    (
        "preprocessor",
        SkinTextPreprocessor(correct_spelling=True)
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=8000,
            sublinear_tf=True,
            min_df=2,
            max_df=0.95
        )
    ),
    (
        "classifier",
        MultiOutputClassifier(
            LinearSVC(
                class_weight="balanced",
                random_state=42
            )
        )
    )
])


# ==========================================
# TRAIN BOTH MODELS
# ==========================================

print("Training Logistic Regression...")
lr_pipeline.fit(X_train, y_train)

print("Training Linear SVM...")
svm_pipeline.fit(X_train, y_train)

print("\nBoth pipelines trained successfully!")


Training Logistic Regression...
Training Linear SVM...

Both pipelines trained successfully!


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

targets = ["oiliness", "dryness", "sensitivity"]

models = {
    "TF-IDF + Logistic Regression": lr_pipeline,
    "TF-IDF + Linear SVM": svm_pipeline
}

results = []

for model_name, model in models.items():

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)

    y_pred = model.predict(X_test)

    for i, target in enumerate(targets):

        y_true_target = y_test[target]
        y_pred_target = y_pred[:, i]

        accuracy = accuracy_score(
            y_true_target, y_pred_target
        )

        print(f"\n--- {target.upper()} ---")
        print(f"Accuracy: {accuracy:.4f}")

        print(
            classification_report(
                y_true_target,
                y_pred_target,
                zero_division=0
            )
        )

        print("Confusion Matrix:")
        labels = model.named_steps["classifier"].estimators_[i].classes_
        print(
            pd.DataFrame(
                confusion_matrix(
                    y_true_target,
                    y_pred_target,
                    labels=labels
                ),
                index=[f"Actual {x}" for x in labels],
                columns=[f"Predicted {x}" for x in labels]
            )
        )

        results.append({
            "Model": model_name,
            "Target": target,
            "Accuracy": accuracy
        })

results_df = pd.DataFrame(results)

print("\n\nSUMMARY OF ACCURACY")
print(
    results_df.pivot(
        index="Target",
        columns="Model",
        values="Accuracy"
    ).round(4)
)



TF-IDF + Logistic Regression

--- OILINESS ---
Accuracy: 0.9056
              precision    recall  f1-score   support

        High       0.91      0.98      0.94       244
         Low       0.88      0.96      0.92       226
      Medium       0.94      0.78      0.85       250

    accuracy                           0.91       720
   macro avg       0.91      0.91      0.90       720
weighted avg       0.91      0.91      0.90       720

Confusion Matrix:
               Predicted High  Predicted Low  Predicted Medium
Actual High               240              0                 4
Actual Low                  0            217                 9
Actual Medium              24             31               195

--- DRYNESS ---
Accuracy: 0.8875
              precision    recall  f1-score   support

        High       0.93      0.91      0.92       240
         Low       0.86      0.98      0.92       238
      Medium       0.87      0.78      0.82       242

    accuracy                    

In [ ]:

import pandas as pd
import numpy as np

def detailed_skin_prediction(text, model, model_name="Model"):

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please enter a non-empty skin description.")

    targets = ["oiliness", "dryness", "sensitivity"]

    print("=" * 75)
    print(f"DETAILED PREDICTION: {model_name}")
    print("=" * 75)

    # ------------------------------------------
    # 1. SHOW PREPROCESSING STEPS
    # ------------------------------------------

    preprocessor = model.named_steps["preprocessor"]
    steps = preprocessor.process_with_steps(text)

    print("\n1. ORIGINAL INPUT")
    print(steps["original"])

    print("\n2. BASIC CLEANING")
    print(steps["basic_cleaning"])

    print("\n3. CONTRACTION EXPANSION")
    print(steps["contraction_expansion"])

    print("\n4. SPELLING CORRECTION")
    print(steps["spelling_correction"])

    print("\n5. POS TAGS AND LEMMAS")
    print(
        pd.DataFrame(steps["pos_tags_and_lemmas"]).to_string(index=False)
    )

    print("\n6. FINAL PREPROCESSED TEXT")
    print(steps["final"])

    # ------------------------------------------
    # 2. SHOW TF-IDF FEATURES
    # ------------------------------------------

    vectorizer = model.named_steps["tfidf"]
    processed_text = steps["final"]

    tfidf_vector = vectorizer.transform([processed_text])

    feature_names = vectorizer.get_feature_names_out()
    values = tfidf_vector.toarray()[0]

    nonzero_indices = np.flatnonzero(values)

    print("\n7. TF-IDF FEATURES")
    print("Vocabulary size:", len(feature_names))
    print("Features matched in this input:", len(nonzero_indices))

    if len(nonzero_indices) > 0:
        top_indices = nonzero_indices[
            np.argsort(values[nonzero_indices])[::-1][:15]
        ]

        feature_table = pd.DataFrame({
            "Feature": feature_names[top_indices],
            "TF-IDF weight": values[top_indices]
        })

        print(feature_table.to_string(index=False))
    else:
        print("No known vocabulary features matched this input.")

    # ------------------------------------------
    # 3. PREDICTIONS AND STRENGTH
    # ------------------------------------------

    classifier = model.named_steps["classifier"]
    predictions = classifier.predict(tfidf_vector)[0]

    print("\n8. PREDICTIONS")
    results = []

    for i, target in enumerate(targets):
        estimator = classifier.estimators_[i]
        predicted_label = predictions[i]
        classes = estimator.classes_

        row = {
            "Target": target.capitalize(),
            "Predicted label": predicted_label
        }

        if hasattr(estimator, "predict_proba"):
            # Logistic Regression: actual model-estimated probabilities.
            probabilities = estimator.predict_proba(tfidf_vector)[0]

            for label, probability in zip(classes, probabilities):
                row[f"P({label})"] = round(float(probability), 4)

            confidence = float(np.max(probabilities))
            row["Strength type"] = "Predicted class probability"
            row["Strength"] = round(confidence, 4)

        elif hasattr(estimator, "decision_function"):
            # Linear SVM: decision scores, NOT probabilities.
            scores = estimator.decision_function(tfidf_vector)

            row["Strength type"] = "SVM decision score"

            if np.ndim(scores) == 1 or len(classes) == 2:
                scores = np.asarray(scores).reshape(-1)
                score = float(scores[0])

                # Binary LinearSVC uses positive scores for classes_[1].
                predicted_score = (
                    score if predicted_label == classes[1] else -score
                )

                row["Predicted class score"] = round(predicted_score, 4)
                row["Decision scores"] = {
                    str(classes[0]): round(-score, 4),
                    str(classes[1]): round(score, 4)
                }
            else:
                scores = np.asarray(scores).reshape(-1)

                row["Predicted class score"] = round(
                    float(scores[np.where(classes == predicted_label)[0][0]]),
                    4
                )

                row["Decision scores"] = {
                    str(label): round(float(score), 4)
                    for label, score in zip(classes, scores)
                }

        results.append(row)

    results_df = pd.DataFrame(results)

    print(results_df.to_string(index=False))

    return {
        "steps": steps,
        "tfidf_features": {
            feature_names[i]: float(values[i])
            for i in nonzero_indices
        },
        "results": results_df
    }


In [ ]:

user_text = input("Describe your skin: ")

lr_details = detailed_skin_prediction(
    user_text,
    lr_pipeline,
    "TF-IDF + Logistic Regression"
)

svm_details = detailed_skin_prediction(
    user_text,
    svm_pipeline,
    "TF-IDF + Linear SVM"
)


In [ ]:

# Inspect descriptions containing strong oiliness or lack of dryness.

patterns = [
    "very oily",
    "extremely oily",
    "never dry",
    "not dry",
    "greasy"
]

for pattern in patterns:
    print("\n" + "=" * 60)
    print("SEARCH:", pattern)

    matching = df[
        df["text"].astype(str).str.contains(
            pattern, case=False, regex=False, na=False
        )
    ]

    if matching.empty:
        print("No matching rows found.")
        continue

    print("Number of matching rows:", len(matching))

    print("\nOiliness labels:")
    print(matching["oiliness"].value_counts())

    print("\nDryness labels:")
    print(matching["dryness"].value_counts())

    print("\nExample rows:")
    print(
        matching[
            ["text", "oiliness", "dryness", "sensitivity"]
        ].head(10).to_string(index=False)
    )
